In [2]:
import pandas as pd
from dotenv import load_dotenv
import os
import time
from openai import OpenAI
import json


In [2]:
data_high = pd.read_json("../data/llm_formatted/high_llm.json")
data_low = pd.read_json("../data/llm_formatted/low_llm.json")
TARGET = "HubSpot"

In [3]:
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## LLM

In [4]:
def llm_sentiment(thread, target, subreddit):

    
    prompt = f'''
        You are analyzing Reddit threads about companies.

        Target company: {target}
        Subreddit: {subreddit}
        Thread: {thread}

        Your task is to analyze this thread with respect to {target}.

        1. Determine whether the thread is relevant to {target}.
        If the thread is from a subreddit dedicated to the company, assume it is about {target}, even if the name is not directly mentioned.
        Otherwise, mark it as relevant only if the discussion clearly focuses on {target}.

        2. Classify the overall sentiment of the thread toward {target} as:
        - "positive"
        - "negative"
        - "neutral"

        Use these guidelines:
        - "positive" - users are clearly satisfied, happy, or say something good about {target}
        - "negative" - users complain, express frustration, or describe problems related to {target}
        - "neutral" - the thread mainly asks questions, shares information, or has no clear opinion

        Questions are usually neutral unless they clearly describe a problem or frustration related to {target}.
        A positive or enthusiastic tone alone does not necessarily mean positive sentiment.
        If there is no clear opinion about {target}, choose "neutral".


        3. Determine whether the thread contains actionable product feedback.
        Set "product_feedback" to true only if the thread contains a clear problem, limitation, complaint, suggestion, or concrete request related to {target}.
        Otherwise set it to false.

        Use the full thread (post + comments). If opinions differ, base your answer on the overall dominant view across the full thread.

        Also provide a short reasoning (one sentence) explaining your decision.


        Return only this JSON without any additional text:
        {{
            "relevant_to_target": true | false,
            "sentiment": "positive" | "negative" | "neutral",
            "product_feedback": true | false,
            "reasoning": "one short explanation"
        }}
    '''
    
    try: 
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=prompt,
        )

        raw = response.output_text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()

        analysis = json.loads(raw)
        return analysis
    
    except json.JSONDecodeError as e:
        print("JSON error on:" + response.output_text)
        return{
            "relevant_to_target": False,
            "sentiment": "neutral",
            "product_feedback": False,
            "reasoning": "ERROR"
        }



In [5]:
import pandas as pd
import time

def run_llm_on_dataset(data, target):
    rows = []
    start_time = time.time()

    for i, row in data.iterrows():
        thread = row["thread_for_llm"]
        subreddit = row["subreddit_name"]

        out = {
            "id": row.get("id"),
            "subreddit_name": row.get("subreddit_name"),
            "num_of_comments": row.get("num_of_comments"),
            "score": row.get("score"),
            "upvote_ratio": row.get("upvote_ratio"),
            "created_utc": row.get("created_utc"),
            "thread_for_llm": thread,
        }

        try:
            llm = llm_sentiment(thread, target, subreddit)

            out.update({
                "relevant_to_target": llm.get("relevant_to_target", False),
                "llm_sentiment": llm.get("sentiment", "neutral"),
                "product_feedback": llm.get("product_feedback", False),
                "reasoning": llm.get("reasoning", "")
            })

        except Exception as e:
            out.update({
                "relevant_to_target": False,
                "llm_sentiment": "neutral",
                "product_feedback": False,
                "reasoning": f"ERROR: {e}"
            })

        rows.append(out)

        if (i + 1) % 50 == 0 or (i + 1) == len(data):
            print(f"{i + 1}/{len(data)}")

    results = pd.DataFrame(rows)

    seconds = time.time() - start_time
    print(f"Aega läks: {seconds:.2f} s")

    return results

In [6]:
results_high = run_llm_on_dataset(data_high, TARGET)
results_low = run_llm_on_dataset(data_low, TARGET)

50/987
100/987
150/987
200/987
250/987
300/987
350/987
400/987
450/987
500/987
550/987
600/987
650/987
700/987
750/987
800/987
850/987
900/987
950/987
987/987
Aega läks: 2116.80 s
50/279
100/279
150/279
200/279
250/279
279/279
Aega läks: 601.06 s


In [7]:
results_high["source_set"] = "high"
results_low["source_set"] = "low"

results = pd.concat([results_high, results_low], ignore_index=True)

In [8]:
import os

os.makedirs("../data/signal_level", exist_ok=True)
results.to_json("../data/signal_level/signal_results.json", orient="records", indent=2, force_ascii=False)
results.to_csv("../data/signal_level/signal_results.csv",index=False,encoding="utf-8")

## Tulemuste analüüsimine

In [32]:
results = pd.read_csv("../data/signal_level/signal_results.csv")
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 1266 entries, 0 to 1265
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  1266 non-null   str    
 1   subreddit_name      1266 non-null   str    
 2   num_of_comments     1266 non-null   int64  
 3   score               1266 non-null   int64  
 4   upvote_ratio        1266 non-null   float64
 5   created_utc         1266 non-null   int64  
 6   thread_for_llm      1266 non-null   str    
 7   relevant_to_target  1266 non-null   bool   
 8   llm_sentiment       1266 non-null   str    
 9   product_feedback    1266 non-null   bool   
 10  reasoning           1266 non-null   str    
 11  source_set          1266 non-null   str    
dtypes: bool(2), float64(1), int64(3), str(6)
memory usage: 101.5 KB


In [33]:
results.head(1)

,id,subreddit_name,num_of_comments,score,upvote_ratio,created_utc,thread_for_llm,relevant_to_target,llm_sentiment,product_feedback,reasoning,source_set
0,1sgkv1i,hubspot,2,1,1.0,1775727274,SUBREDDIT: hubspot\n\nPOST [post_id_1sgkv1i]\n...,True,neutral,True,The thread discusses an app addressing limitat...,high


In [34]:
high = results[results["source_set"] == "high"]
low = results[results["source_set"] == "low"]

In [35]:
def compute_metrics(df):
    return {
        "Postituste arv": len(df),
        "Kommentaaride arv keskmiselt": round(df["num_of_comments"].mean(), 1),
        "Negatiivne": round((df["llm_sentiment"] == "negative").mean() * 100, 1),
        "Neutraalne": round((df["llm_sentiment"] == "neutral").mean() * 100, 1),
        "Positiivne": round((df["llm_sentiment"] == "positive").mean() * 100, 1),
        "Sisuline tagasiside": round(df["product_feedback"].mean() * 100, 1),
        "Asjakohane": round(df["relevant_to_target"].mean() * 100, 1),
        "Väärtuslik (negatiivne & sisuline &  asjakohane)" : round((
            (df["relevant_to_target"]) &
            (df["product_feedback"]) &
            (df["llm_sentiment"] == "negative")
        ).mean() * 100, 1)
    }

In [36]:
comparison = pd.DataFrame([
    compute_metrics(high),
    compute_metrics(low)
], index=["Kõrge signaalitasemega", "Madal signaalitasemega"])

comparison

,Postituste arv,Kommentaaride arv keskmiselt,Negatiivne,Neutraalne,Positiivne,Sisuline tagasiside,Asjakohane,Väärtuslik (negatiivne & sisuline & asjakohane)
Kõrge signaalitasemega,987,8.3,17.6,76.7,5.7,76.3,100.0,17.6
Madal signaalitasemega,279,16.5,21.9,72.0,6.1,49.1,93.9,20.8


In [ ]:
#pd.set_option('display.max_colwidth', None)

In [46]:
low[low["relevant_to_target"] == False]

id    subreddit_name  num_of_comments  score  upvote_ratio  \
1020  1s9cq6v               CRM               17     10          0.92   
1096  1pc7ls2               CRM                2      0          0.33   
1113  1olfzro               CRM                8      5          0.73   
1134  1s3b6bs         techsales               19     19          0.96   
1136  1rrqjvm         techsales                8      2          1.00   
1157  1rv5ke8  DigitalMarketing                3      1          1.00   
1185  1q5cl7y  DigitalMarketing                3      1          1.00   
1187  1pzjunz  DigitalMarketing                3      1          1.00   
1198  1p1w6y3  DigitalMarketing                5      1          1.00   
1203  1oka0qi  DigitalMarketing               10     14          0.95   
1216  1rz1hac      Entrepreneur               19      4          0.83   
1223  1qawgja      Entrepreneur                1      1          1.00   
1225  1pov0az      Entrepreneur               12      2          1.00   
1242  1rqz2r5      b2bmarketing               29      8          0.58   
1243  1rq65ks      b2bmarketing                7      3          1.00   
1252  1qub4ty      b2bmarketing                6      5          1.00   
1263  1ovefl1      b2bmarketing                4      0          0.13   

      created_utc  \
1020   1775025307   
1096   1764679817   
1113   1761974881   
1134   1774446325   
1136   1773321583   
1157   1773656143   
1185   1767686101   
1187   1767105787   
1198   1763620587   
1203   1761854397   
1216   1774025228   
1223   1768228445   
1225   1765973542   
1242   1773247675   
1243   1773169670   
1252   1770074134   
1263   1762974779   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [44]:
for i in low[low["relevant_to_target"] == False]["thread_for_llm"]:
    print(i)

SUBREDDIT: CRM

POST [post_id_1s9cq6v]
TITLE: What’s something you only realized after implementing a CRM that you wish you knew before?
BODY: After so many discussions and 1:1 with my sales team regarding the slow down in the revenue. The maximum reasons are crm data maintenance consumes maximum of time in the day, Maximum time in working in proposals. They want to AI support on data updates include the meeting notes, auto update in contacts and companies, supports the automated follow-up reminders. Since we have recently shifted to zoho from another crm. without discussing and understanding the requirements of the team, we bought the crm. suggest me the solutions and with less pricey. i'm trying to avoid the hubspot because of cost. How you have finalized the crm and improved your teams efficiency. Would you suggest solution for this.

COMMENTS:

COMMENT [comment_id_odnsn87]
parent_id: post_1s9cq6v
depth: 0
score: 3
text: Biggest realization: the CRM doesn’t fix a broken process (but

In [21]:
high_rel = high[high["relevant_to_target"]]
low_rel = low[low["relevant_to_target"]]

comparison_rel = pd.DataFrame([
    compute_metrics(high_rel),
    compute_metrics(low_rel)
], index=["Kõrge (relevant)", "Madal (relevant)"])

comparison_rel

,Postituste arv,Kommentaaride arv keskmiselt,Negatiivne,Neutraalne,Positiivne,Sisuline tagasiside,Asjakohane,Väärtuslik (negatiivne & sisuline & asjakohaneant)
Kõrge (relevant),987,8.3,17.6,76.7,5.7,76.3,100.0,17.6
Madal (relevant),262,16.9,23.3,70.2,6.5,51.5,100.0,22.1


In [16]:
valuable = results[
    (results["relevant_to_target"]) &
    (results["product_feedback"]) &
    (results["llm_sentiment"] == "negative")
]

valuable[["thread_for_llm", "reasoning"]].head(5)

,thread_for_llm,reasoning
20,SUBREDDIT: hubspot\n\nPOST [post_id_1se3d6w]\n...,The thread discusses frustrations with HubSpot...
23,SUBREDDIT: hubspot\n\nPOST [post_id_1sdni3i]\n...,The post and comments describe frustration and...
41,SUBREDDIT: hubspot\n\nPOST [post_id_1s9wle5]\n...,The thread revolves around performance issues ...
46,SUBREDDIT: hubspot\n\nPOST [post_id_1s994et]\n...,The thread discusses issues with HubSpot being...
57,SUBREDDIT: hubspot\n\nPOST [post_id_1s5w14v]\n...,The thread discusses user frustration with Hub...
